# JobToken.io

### Install dependencies

In [1]:
# If running in Colab, run this cell once per session.
!pip -q install python-jobspy pandas numpy pyarrow seaborn

### Imports

In [2]:
from pathlib import Path
from datetime import datetime
import pandas as pd
import re

# JobSpy import can vary by version. If your notebook already imports correctly, keep yours.
from jobspy import scrape_jobs

In [3]:
# Project root = parent of the notebooks folder
PROJECT_ROOT = Path.cwd().parent  # assumes you run notebook from /notebooks
DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

DATA_RAW.mkdir(parents=True, exist_ok=True)
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

In [4]:
# Double check path
print("CWD:", Path.cwd())
print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_RAW:", DATA_RAW.resolve())
print("DATA_PROCESSED:", DATA_PROCESSED.resolve())

CWD: /Users/Jordan/Documents/USD/Spring 2026/ADS 509/Final Project/notebooks
PROJECT_ROOT: /Users/Jordan/Documents/USD/Spring 2026/ADS 509/Final Project
DATA_RAW: /Users/Jordan/Documents/USD/Spring 2026/ADS 509/Final Project/data/raw
DATA_PROCESSED: /Users/Jordan/Documents/USD/Spring 2026/ADS 509/Final Project/data/processed


### Run Configuration and Timestamp

In [5]:
# =========================
# Run configuration and Timestamp
# =========================
SEARCH_TERM = "data scientist"
RESULTS_PER_SITE = 100
LOCATION = "San Francisco Bay Area"  # e.g., "San Francisco Bay Area" if you want to match the original notebook

SITES = ["linkedin", "indeed"]

RUN_TS = datetime.now().strftime("%Y%m%d_%H%M%S")

### Helper Functions (Cleaning + Standardization) 

In [6]:
def clean_text(s: str) -> str:
    if pd.isna(s):
        return ""
    s = str(s)
    s = re.sub(r"<[^>]+>", " ", s)      # remove html
    s = re.sub(r"\s+", " ", s).strip()  # normalize whitespace
    return s

def standardize_jobs(df: pd.DataFrame, source: str, search_term: str) -> pd.DataFrame:
    df = df.copy()

    # Ensure columns exist (JobSpy output can vary)
    for col in ["title", "company", "location", "date_posted", "job_type", "description", "job_url"]:
        if col not in df.columns:
            df[col] = None

    df["description"] = df["description"].apply(clean_text)
    df["source"] = source
    df["search_term"] = search_term

    # text column used by downstream NLP
    df["text"] = (df["title"].fillna("").astype(str) + " " + df["description"].fillna("").astype(str)).str.lower().str.strip()

    # Final column order (stable)
    cols = ["source", "search_term", "title", "company", "location", "date_posted", "job_type", "description", "job_url", "text"]
    return df[cols]

### Scrape LinkedIn and Save

In [7]:
jobs_linkedin_raw = scrape_jobs(
    site_name=["linkedin"],
    search_term=SEARCH_TERM,
    location=LOCATION,
    results_wanted=RESULTS_PER_SITE,
    linkedin_fetch_description=True,
    verbose=1,
)

df_linkedin = standardize_jobs(jobs_linkedin_raw, source="linkedin", search_term=SEARCH_TERM)

linkedin_path = DATA_RAW / f"jobs_linkedin_{RUN_TS}.csv"
df_linkedin.to_csv(linkedin_path, index=False)

print("LinkedIn rows:", len(df_linkedin))
print("Saved:", linkedin_path.resolve())
df_linkedin.head(3)

2026-02-23 21:25:21,137 - INFO - JobSpy:Linkedin - finished scraping


LinkedIn rows: 100
Saved: /Users/Jordan/Documents/USD/Spring 2026/ADS 509/Final Project/data/raw/jobs_linkedin_20260223_212315.csv


,source,search_term,title,company,location,date_posted,job_type,description,job_url,text
0,linkedin,data scientist,Software Engineer I,Twitch,"San Francisco, CA",2026-02-22,fulltime,**About Us** Twitch is the world’s biggest liv...,https://www.linkedin.com/jobs/view/4344081079,software engineer i **about us** twitch is the...
1,linkedin,data scientist,"Machine Learning/AI Scientist Intern (PhD), 2026",Netflix,"Los Gatos, CA",2026-02-22,internship,Netflix is one of the world's leading entertai...,https://www.linkedin.com/jobs/view/4312233312,"machine learning/ai scientist intern (phd), 20..."
2,linkedin,data scientist,"Software Engineer, NLP/Machine Learning",WisdomAI,"San Mateo, CA",2026-02-22,fulltime,**About Wisdom** WisdomAI has the mission to p...,https://www.linkedin.com/jobs/view/4376331886,"software engineer, nlp/machine learning **abou..."


### Scrape Indeed and Save

In [8]:
jobs_indeed_raw = scrape_jobs(
    site_name=["indeed"],
    search_term=SEARCH_TERM,
    location=LOCATION,
    results_wanted=RESULTS_PER_SITE,
    country_indeed="USA",
    verbose=1,
)

df_indeed = standardize_jobs(jobs_indeed_raw, source="indeed", search_term=SEARCH_TERM)

indeed_path = DATA_RAW / f"jobs_indeed_{RUN_TS}.csv"
df_indeed.to_csv(indeed_path, index=False)

print("Indeed rows:", len(df_indeed))
print("Saved:", indeed_path.resolve())
df_indeed.head(3)

Indeed rows: 100
Saved: /Users/Jordan/Documents/USD/Spring 2026/ADS 509/Final Project/data/raw/jobs_indeed_20260223_212315.csv


,source,search_term,title,company,location,date_posted,job_type,description,job_url,text
0,indeed,data scientist,"Data Scientist, Marketing Innovation",OpenAI,"San Francisco, CA, US",2026-02-23,fulltime,"Careers Data Scientist, Marketing Innovation D...",https://www.indeed.com/viewjob?jk=50da1a7d4cd0...,"data scientist, marketing innovation careers d..."
1,indeed,data scientist,Sr. AI/ML Engineer-4,Realign,"San Francisco, CA, US",2026-02-23,contract,"San Francisco, California 94016 Posted Februar...",https://www.indeed.com/viewjob?jk=667798be4f13...,"sr. ai/ml engineer-4 san francisco, california..."
2,indeed,data scientist,Data Scientist,Pacific Community Ventures,"Oakland, CA, US",2026-02-23,fulltime,**About Us** PCV is a nonprofit community deve...,https://www.indeed.com/viewjob?jk=0447d34a5a96...,data scientist **about us** pcv is a nonprofit...


### Combine into one dataset

In [9]:
import duckdb

con = duckdb.connect()

linkedin_file = (DATA_RAW / f"jobs_linkedin_{RUN_TS}.csv").as_posix()
indeed_file   = (DATA_RAW / f"jobs_indeed_{RUN_TS}.csv").as_posix()

query = f"""
CREATE OR REPLACE TABLE combined_jobs AS
SELECT * FROM read_csv_auto('{linkedin_file}')
UNION ALL
SELECT * FROM read_csv_auto('{indeed_file}');
"""

con.execute(query)

combined = con.execute("SELECT * FROM combined_jobs").df()
print("Combined rows (this run only):", len(combined))
combined["source"].value_counts()
combined.head(5)

Combined rows (this run only): 200


,source,search_term,title,company,location,date_posted,job_type,description,job_url,text
0,linkedin,data scientist,Software Engineer I,Twitch,"San Francisco, CA",2026-02-22,fulltime,**About Us** Twitch is the world’s biggest liv...,https://www.linkedin.com/jobs/view/4344081079,software engineer i **about us** twitch is the...
1,linkedin,data scientist,"Machine Learning/AI Scientist Intern (PhD), 2026",Netflix,"Los Gatos, CA",2026-02-22,internship,Netflix is one of the world's leading entertai...,https://www.linkedin.com/jobs/view/4312233312,"machine learning/ai scientist intern (phd), 20..."
2,linkedin,data scientist,"Software Engineer, NLP/Machine Learning",WisdomAI,"San Mateo, CA",2026-02-22,fulltime,**About Wisdom** WisdomAI has the mission to p...,https://www.linkedin.com/jobs/view/4376331886,"software engineer, nlp/machine learning **abou..."
3,linkedin,data scientist,"Data Science, Intern",Tatari,"San Francisco, CA",2026-02-21,internship,Tatari is on a mission to revolutionize TV adv...,https://www.linkedin.com/jobs/view/4374689270,"data science, intern tatari is on a mission to..."
4,linkedin,data scientist,"Data Scientist- Business Planning, Pricing",AMD,"San Jose, CA",2026-02-21,fulltime,**WHAT YOU DO AT AMD CHANGES EVERYTHING** At A...,https://www.linkedin.com/jobs/view/4356712751,"data scientist- business planning, pricing **w..."


In [10]:
# Which files were combined with timestamp
print("Combining files:")
print(" -", linkedin_file)
print(" -", indeed_file)

Combining files:
 - /Users/Jordan/Documents/USD/Spring 2026/ADS 509/Final Project/data/raw/jobs_linkedin_20260223_212315.csv
 - /Users/Jordan/Documents/USD/Spring 2026/ADS 509/Final Project/data/raw/jobs_indeed_20260223_212315.csv


### Deduplicate the combined dataset

In [11]:
combined = con.execute("SELECT * FROM combined_jobs").df()

# De-Duplecate function
import hashlib

def make_signature_row(row) -> str:
    """
    Fallback signature when job_url is missing.
    Uses stable fields and a short hash of description.
    """
    title = str(row.get("title", "") or "").strip().lower()
    company = str(row.get("company", "") or "").strip().lower()
    location = str(row.get("location", "") or "").strip().lower()
    desc = str(row.get("description", "") or "").strip().lower()

    # hash only first chunk of description to keep it stable + fast
    desc_hash = hashlib.md5(desc[:500].encode("utf-8")).hexdigest()

    return f"{title}|{company}|{location}|{desc_hash}"

def dedupe_jobs(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # Normalize url if present
    if "job_url" in df.columns:
        df["job_url_norm"] = (
            df["job_url"].fillna("")
            .astype(str)
            .str.strip()
        )
    else:
        df["job_url_norm"] = ""

    # Primary dedupe key: job_url when available
    has_url = df["job_url_norm"].str.len() > 0

    # Split into url and non-url sets
    df_url = df[has_url].copy()
    df_nourl = df[~has_url].copy()

    before = len(df)

    if len(df_url) > 0:
        df_url = df_url.drop_duplicates(subset=["job_url_norm"])

    if len(df_nourl) > 0:
        df_nourl["signature"] = df_nourl.apply(make_signature_row, axis=1)
        df_nourl = df_nourl.drop_duplicates(subset=["signature"])

    df_out = pd.concat([df_url, df_nourl], ignore_index=True)

    after = len(df_out)
    print(f"Deduped rows: {before} -> {after} (removed {before-after})")

    # Cleanup helper cols
    drop_cols = [c for c in ["job_url_norm", "signature"] if c in df_out.columns]
    df_out = df_out.drop(columns=drop_cols, errors="ignore")

    return df_out

In [12]:
combined_dedup = dedupe_jobs(combined)
combined_dedup["source"].value_counts()

Deduped rows: 200 -> 200 (removed 0)


source
linkedin    100
indeed      100
Name: count, dtype: int64

### Enforce 100 per site

In [13]:
def cap_per_source(df: pd.DataFrame, cap: int = 100) -> pd.DataFrame:
    df = df.copy()

    # If date_posted exists, keep newest; otherwise keep first occurrences
    if "date_posted" in df.columns:
        df["date_posted_tmp"] = pd.to_datetime(df["date_posted"], errors="coerce")
        df = df.sort_values(by=["source", "date_posted_tmp"], ascending=[True, False])
        df = df.groupby("source").head(cap).drop(columns=["date_posted_tmp"], errors="ignore")
    else:
        df = df.groupby("source").head(cap)

    return df.reset_index(drop=True)

final_df = cap_per_source(combined_dedup, cap=100)

print("Final rows:", len(final_df))
print(final_df["source"].value_counts())

Final rows: 200
source
indeed      100
linkedin    100
Name: count, dtype: int64


### Save into one dataset

In [14]:
out_combined = DATA_PROCESSED / "jobs_combined.csv"
final_df.to_csv(out_combined, index=False)
print("Saved:", out_combined.resolve())

Saved: /Users/Jordan/Documents/USD/Spring 2026/ADS 509/Final Project/data/processed/jobs_combined.csv


---
In this notebook, we constructed a reproducible data collection pipeline to build a real-world dataset of Data Scientist job postings.

First, we configured role-based search parameters and used the JobSpy library to programmatically scrape job listings from LinkedIn and Indeed. For each source, we retrieved 100 postings filtered by search term and location. The raw outputs were saved as timestamped CSV files in the data/raw/ directory to preserve a record of each scraping run.

Next, we standardized the schema across both sources to ensure structural consistency. This included:
- Cleaning HTML artifacts and formatting inconsistencies in job descriptions
- Creating a unified set of columns (source, search_term, title, company, location, date_posted, job_type, description, job_url, text)
- Constructing a combined lowercase text field to support downstream NLP analysis

We then combined only the files generated in the current run using DuckDB to create a unified dataset. This ensures that each execution of the notebook produces a deterministic dataset rather than accumulating historical runs.

To improve data quality, we implemented a two-stage deduplication process:
1. Primary deduplication using job URLs when available
2. Fallback deduplication using a stable signature built from title, company, location, and a hashed description

Finally, we enforced a balanced dataset by capping the number of listings to 100 per source. This guarantees a consistent final dataset of 200 postings (100 LinkedIn and 100 Indeed).

---